# Extract diagnosis dates and review flags

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import pandas as pd

from smmnlp.config import ModelConfig
from smmnlp.inference import generate_responses
from smmnlp.modeling import LocalLLM
from smmnlp.paths import ProjectPaths
from smmnlp.prompts import Task

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.
from smmnlp.preprocessing import add_date_extractions


## Patient-level hx sMM Date extraction 1. Find patients manually labeled as positive for SMM (n=199, n=85) 2. Identify the first positive SMM encournter 3. Identify the first positive SMM encournter manually labeled as hx (n=77, n=37) 4.1 Extract the date of dx 4.2 Determine whether the extracted date of dx needs human review


In [ ]:
generator = LocalLLM.from_pretrained(ModelConfig(paths.model_path))

In [ ]:
readfile_name = paths.data("Update/Data/Training/Outcome3_training_dataset.json")
notes = pd.read_json(readfile_name)
note_count = len(notes)

In [ ]:
notes.head(5)

In [ ]:
responses = generate_responses(notes["NoteText"].tolist(), generator, Task.DATE)

In [ ]:
notes["rawoutput"] = responses

In [ ]:
responses

In [ ]:
notes = add_date_extractions(notes)


In [ ]:
notes.head(5)

In [ ]:
savefile_name = paths.data("Update/ExtraLLMRun/phase3_fewshot_dxdate_human_updtraining.xlsx")
notes.to_excel(savefile_name, engine="xlsxwriter", sheet_name="Sheet1", index=False)